<a href="https://colab.research.google.com/github/PattiAlves/Modelagem_IA/blob/main/Notebooks/2.%20TREINAMENTO%20DO%20MODELO%20E%20TESTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **2. TREINAMENTO DO MODELO E TESTE**

### **2.1 Divisão da base de dados consolidade em 4 k-folds**

In [2]:
import json
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Carregar a tabela unificada apenas com variáveis de 10 minutos
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
caminho_tabela_10min = os.path.join(
    pasta_consolidados, 'tabela_unificada_apenas_10min_ate_20170916_1840.csv'
)

pasta_splits = os.path.join(pasta_consolidados, 'SPLITS_70_30_4FOLDS')
os.makedirs(pasta_splits, exist_ok=True)

df = pd.read_csv(caminho_tabela_10min)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = (
    df.dropna(subset=['datetime'])
    .sort_values('datetime')
    .reset_index(drop=True)
)

features_10min = ['chuva_10min_mm', 'mare_10min_m']
alvo_10min = 'NA_lagoa_10min_m'

# Filtra apenas registros reais válidos (sem NaN), sem interpolar ou alterar valores
df_valido = df.dropna(subset=features_10min + [alvo_10min]).reset_index(drop=True)
n_total = len(df_valido)

# 2. Separar 70% para o ciclo k-fold e 30% como holdout reservado
idx_corte_70 = int(n_total * 0.70)
idx_treino_70 = np.arange(0, idx_corte_70)
idx_holdout_30 = np.arange(idx_corte_70, n_total)

# 3. Divisão em 4 blocos contíguos (0-25%, 25-50%, 50-75%, 75-100%) dentro dos 70%
#    Em cada fold, 1 bloco é testing/validation fold e os outros 3 blocos são training fold
kf_4_blocos = KFold(n_splits=4, shuffle=False)
folds_4 = {}

for fold_num, (tr_pos, te_pos) in enumerate(
    kf_4_blocos.split(idx_treino_70), start=1
):
  tr_idx = idx_treino_70[tr_pos].tolist()
  te_idx = idx_treino_70[te_pos].tolist()
  folds_4[f'fold_{fold_num}'] = {
      'train_idx': tr_idx,
      'test_fold_idx': te_idx,
  }

# 4. Salvar partições em JSON para reuso idêntico em outros modelos
caminho_json = os.path.join(pasta_splits, 'splits_70_holdout30_4folds_blocos.json')
with open(caminho_json, 'w', encoding='utf-8') as f:
  json.dump(
      {
          'n_total_validos': int(n_total),
          'proporcao_treino': 0.70,
          'proporcao_holdout': 0.30,
          'k_folds': 4,
          'divisao_em_blocos_contiguos': True,
          'treino_70_idx': idx_treino_70.tolist(),
          'holdout_30_idx': idx_holdout_30.tolist(),
          'folds_4': folds_4,
      },
      f,
      indent=2,
  )

# 5. Salvar CSV de referência com a identificação de cada registro
df_ref = df_valido[['datetime'] + features_10min + [alvo_10min]].copy()
df_ref['subconjunto_principal'] = np.where(
    df_ref.index < idx_corte_70, 'treino_70_kfold', 'holdout_30'
)
df_ref['bloco_25pct_no_treino70'] = np.nan
df_ref.loc[idx_treino_70, 'bloco_25pct_no_treino70'] = (
    pd.qcut(np.arange(len(idx_treino_70)), q=4, labels=[1, 2, 3, 4]).astype(int)
)

caminho_csv_ref = os.path.join(
    pasta_splits, 'referencia_divisao_70_30_4folds_blocos.csv'
)
df_ref.to_csv(caminho_csv_ref, index=False)

print('=== DIVISÃO EM 4 K-FOLDS (BLOCOS CONTÍGUOS) CONCLUÍDA ===')
print(f'Total de registros válidos: {n_total}')
print(
    f'Bloco principal 70% (k-fold): {len(idx_treino_70)} registros |'
    f' {df_valido["datetime"].iloc[idx_treino_70[0]]} até'
    f' {df_valido["datetime"].iloc[idx_treino_70[-1]]}'
)
print(
    f'Holdout 30% (reservado):      {len(idx_holdout_30)} registros |'
    f' {df_valido["datetime"].iloc[idx_holdout_30[0]]} até'
    f' {df_valido["datetime"].iloc[idx_holdout_30[-1]]}\n'
)

for fold_num in range(1, 5):
  info = folds_4[f'fold_{fold_num}']
  te_i = info['test_fold_idx']
  print(
      f'Fold {fold_num} -> treino: {len(info["train_idx"])} | bloco teste:'
      f' {len(te_i)} ({df_valido["datetime"].iloc[te_i[0]]} até'
      f' {df_valido["datetime"].iloc[te_i[-1]]})'
  )

print(f'\nArquivos salvos em:\n  • {caminho_json}\n  • {caminho_csv_ref}')

=== DIVISÃO EM 4 K-FOLDS (BLOCOS CONTÍGUOS) CONCLUÍDA ===
Total de registros válidos: 205879
Bloco principal 70% (k-fold): 144115 registros | 2013-02-28 16:20:00 até 2015-12-15 02:10:00
Holdout 30% (reservado):      61764 registros | 2015-12-15 02:20:00 até 2017-09-16 18:40:00

Fold 1 -> treino: 108086 | bloco teste: 36029 (2013-02-28 16:20:00 até 2013-11-12 08:00:00)
Fold 2 -> treino: 108086 | bloco teste: 36029 (2013-11-12 08:10:00 até 2014-07-28 21:00:00)
Fold 3 -> treino: 108086 | bloco teste: 36029 (2014-07-28 21:10:00 até 2015-04-06 04:40:00)
Fold 4 -> treino: 108087 | bloco teste: 36028 (2015-04-06 04:50:00 até 2015-12-15 02:10:00)

Arquivos salvos em:
  • /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/SPLITS_70_30_4FOLDS/splits_70_holdout30_4folds_blocos.json
  • /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/SPLITS_70_30_4FOLDS/referencia_divisao_70_30_4folds_blocos.csv


### **2.2 Treinamento e métricas de validação**

### **2.2.1 Random Forest Regressor (banco de dados com variável de 10min)**

In [3]:
import json
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos de entrada e saída
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_splits = os.path.join(pasta_consolidados, 'SPLITS_70_30_4FOLDS')
pasta_comparacao = os.path.join(
    pasta_consolidados, 'VALIDACAO_COMPARACAO_MODELOS'
)
os.makedirs(pasta_comparacao, exist_ok=True)

caminho_tabela_10min = os.path.join(
    pasta_consolidados, 'tabela_unificada_apenas_10min_ate_20170916_1840.csv'
)
caminho_splits_json = os.path.join(
    pasta_splits, 'splits_70_holdout30_4folds_blocos.json'
)

NOME_MODELO = 'RandomForestRegressor_10min'


def calcular_metricas_regressao(y_real, y_pred):
  """Calcula r (Pearson), R², SSE e MSE apenas sobre valores reais válidos."""
  y_real = np.asarray(y_real, dtype=float)
  y_pred = np.asarray(y_pred, dtype=float)

  residuos = y_real - y_pred
  sse = float(np.sum(residuos**2))
  mse = float(mean_squared_error(y_real, y_pred))
  r2 = float(r2_score(y_real, y_pred))

  if np.std(y_real) == 0 or np.std(y_pred) == 0:
    r = np.nan
  else:
    r = float(np.corrcoef(y_real, y_pred)[0, 1])

  return {'r': r, 'R2': r2, 'SSE': sse, 'MSE': mse}


# 2. Carregar dados reais válidos (sem interpolação) e divisões salvas
df = pd.read_csv(caminho_tabela_10min)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = (
    df.dropna(subset=['datetime'])
    .sort_values('datetime')
    .reset_index(drop=True)
)

features_10min = ['chuva_10min_mm', 'mare_10min_m']
alvo_10min = 'NA_lagoa_10min_m'

df_valido = df.dropna(subset=features_10min + [alvo_10min]).reset_index(drop=True)
X = df_valido[features_10min].values
y = df_valido[alvo_10min].values
datas = df_valido['datetime']

with open(caminho_splits_json, 'r', encoding='utf-8') as f:
  splits = json.load(f)

treino_70_idx = splits['treino_70_idx']
holdout_30_idx = splits['holdout_30_idx']
folds_4 = splits['folds_4']

# 3. Treinamento e validação nos 4 k-folds contíguos (dentro dos 70%)
linhas_folds = []
modelos_folds = {}

for fold_num in range(1, 5):
  info_fold = folds_4[f'fold_{fold_num}']
  tr_idx = info_fold['train_idx']
  te_idx = info_fold['test_fold_idx']

  X_tr, y_tr = X[tr_idx], y[tr_idx]
  X_te, y_te = X[te_idx], y[te_idx]

  rf_fold = RandomForestRegressor(
      n_estimators=200,
      max_depth=16,
      min_samples_leaf=5,
      n_jobs=-1,
      random_state=42,
  )
  rf_fold.fit(X_tr, y_tr)
  modelos_folds[f'fold_{fold_num}'] = rf_fold

  pred_te = rf_fold.predict(X_te)
  m = calcular_metricas_regressao(y_te, pred_te)

  linhas_folds.append({
      'modelo': NOME_MODELO,
      'etapa': f'fold_{fold_num}',
      'n_amostras': len(te_idx),
      'r': round(m['r'], 4) if pd.notna(m['r']) else np.nan,
      'R2': round(m['R2'], 4),
      'SSE': round(m['SSE'], 4),
      'MSE': round(m['MSE'], 6),
  })

df_folds = pd.DataFrame(linhas_folds)

media_4folds = {
    'modelo': NOME_MODELO,
    'etapa': 'media_4folds',
    'n_amostras': int(df_folds['n_amostras'].mean()),
    'r': round(float(df_folds['r'].mean()), 4),
    'R2': round(float(df_folds['R2'].mean()), 4),
    'SSE': round(float(df_folds['SSE'].mean()), 4),
    'MSE': round(float(df_folds['MSE'].mean()), 6),
}

# 4. Treino em todo o bloco de 70% e avaliação no Holdout (30%)
X_tr70, y_tr70 = X[treino_70_idx], y[treino_70_idx]
X_ho30, y_ho30 = X[holdout_30_idx], y[holdout_30_idx]

rf_70 = RandomForestRegressor(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
rf_70.fit(X_tr70, y_tr70)

pred_ho30 = rf_70.predict(X_ho30)
m_ho = calcular_metricas_regressao(y_ho30, pred_ho30)

linha_holdout = {
    'modelo': NOME_MODELO,
    'etapa': 'holdout_30',
    'n_amostras': len(y_ho30),
    'r': round(m_ho['r'], 4) if pd.notna(m_ho['r']) else np.nan,
    'R2': round(m_ho['R2'], 4),
    'SSE': round(m_ho['SSE'], 4),
    'MSE': round(m_ho['MSE'], 6),
}

# 5. Tabela consolidada de avaliação deste modelo
df_avaliacao_rf = pd.concat(
    [df_folds, pd.DataFrame([media_4folds, linha_holdout])], ignore_index=True
)

print('=== VALIDAÇÃO DO RANDOM FOREST REGRESSOR (4 FOLDS + HOLDOUT 30%) ===')
print(df_avaliacao_rf.to_string(index=False))

# 6. Atualizar / criar tabela comparativa geral entre modelos
caminho_avaliacao_rf = os.path.join(
    pasta_comparacao, 'avaliacao_4folds_holdout30_rf.csv'
)
caminho_comparacao_geral = os.path.join(
    pasta_comparacao, 'comparacao_geral_modelos.csv'
)
caminho_modelo_rf = os.path.join(
    pasta_comparacao, 'modelo_rf_10min_treino70.joblib'
)

df_avaliacao_rf.to_csv(caminho_avaliacao_rf, index=False)
joblib.dump(rf_70, caminho_modelo_rf)

if os.path.exists(caminho_comparacao_geral):
  df_comp_ant = pd.read_csv(caminho_comparacao_geral)
  df_comp_ant = df_comp_ant[df_comp_ant['modelo'] != NOME_MODELO]
  df_comp_geral = pd.concat(
      [df_comp_ant, df_avaliacao_rf], ignore_index=True
  )
else:
  df_comp_geral = df_avaliacao_rf.copy()

df_comp_geral.to_csv(caminho_comparacao_geral, index=False)

# Resumo lado a lado (média 4 folds vs. holdout 30%) para comparação rápida
resumo_pivot = df_comp_geral[
    df_comp_geral['etapa'].isin(['media_4folds', 'holdout_30'])
].pivot(index='modelo', columns='etapa', values=['r', 'R2', 'SSE', 'MSE'])

print('\n=== COMPARAÇÃO ENTRE MODELOS (MÉDIA 4 FOLDS VS. HOLDOUT 30%) ===')
print(resumo_pivot.to_string())
print(f'\nArquivos salvos em: {pasta_comparacao}')

=== VALIDAÇÃO DO RANDOM FOREST REGRESSOR (4 FOLDS + HOLDOUT 30%) ===
                     modelo        etapa  n_amostras      r      R2       SSE      MSE
RandomForestRegressor_10min       fold_1       36029 0.1563 -0.5576 1181.5827 0.032795
RandomForestRegressor_10min       fold_2       36029 0.1817 -0.0160  602.9727 0.016736
RandomForestRegressor_10min       fold_3       36029 0.1263 -0.0667  408.3064 0.011333
RandomForestRegressor_10min       fold_4       36028 0.1386 -0.8655  991.1686 0.027511
RandomForestRegressor_10min media_4folds       36028 0.1507 -0.3765  796.0076 0.022094
RandomForestRegressor_10min   holdout_30       61764 0.1297 -0.0085 2639.1462 0.042730

=== COMPARAÇÃO ENTRE MODELOS (MÉDIA 4 FOLDS VS. HOLDOUT 30%) ===
                                     r                      R2                     SSE                     MSE             
etapa                       holdout_30 media_4folds holdout_30 media_4folds holdout_30 media_4folds holdout_30 media_4folds
modelo  

### **2.2.2 Modelo Random Forest Regressor (banco de dados com variáveis de acumulados e defasagem)**

In [1]:
import json
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos de entrada e saída
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_splits = os.path.join(pasta_consolidados, 'SPLITS_70_30_4FOLDS')
pasta_comparacao = os.path.join(
    pasta_consolidados, 'VALIDACAO_COMPARACAO_MODELOS'
)
os.makedirs(pasta_comparacao, exist_ok=True)

caminho_tabela_10min = os.path.join(
    pasta_consolidados, 'tabela_unificada_apenas_10min_ate_20170916_1840.csv'
)
caminho_splits_json = os.path.join(
    pasta_splits, 'splits_70_holdout30_4folds_blocos.json'
)

NOME_MODELO = 'RandomForestRegressor_10min_acum_lags'


def calcular_metricas_regressao(y_real, y_pred):
  """Calcula r (Pearson), R², SSE e MSE apenas sobre valores reais válidos."""
  y_real = np.asarray(y_real, dtype=float)
  y_pred = np.asarray(y_pred, dtype=float)

  residuos = y_real - y_pred
  sse = float(np.sum(residuos**2))
  mse = float(mean_squared_error(y_real, y_pred))
  r2 = float(r2_score(y_real, y_pred))

  if np.std(y_real) == 0 or np.std(y_pred) == 0:
    r = np.nan
  else:
    r = float(np.corrcoef(y_real, y_pred)[0, 1])

  return {'r': r, 'R2': r2, 'SSE': sse, 'MSE': mse}


# 2. Carregar tabela unificada de 10 min e garantir grade contínua para acumulados/lags
df = pd.read_csv(caminho_tabela_10min)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = (
    df.dropna(subset=['datetime'])
    .sort_values('datetime')
    .drop_duplicates(subset=['datetime'], keep='last')
    .set_index('datetime')
)

grade_10min = pd.date_range(
    start=df.index.min(), end=df.index.max(), freq='10min'
)
df_grade = df.reindex(grade_10min)

chuva = pd.to_numeric(df_grade['chuva_10min_mm'], errors='coerce')
mare = pd.to_numeric(df_grade['mare_10min_m'], errors='coerce')
alvo = pd.to_numeric(df_grade['NA_lagoa_10min_m'], errors='coerce')

# 3. Engenharia de variáveis: instantâneas + acumulados de chuva + defasagens (lags)
feat = pd.DataFrame(index=grade_10min)
feat['chuva_10min_mm'] = chuva
feat['mare_10min_m'] = mare

# Acumulados antecedentes de chuva (passos de 10 min: 30min=3, 1h=6, 3h=18, 6h=36, 12h=72, 24h=144)
janelas_chuva = {
    'chuva_acum_30min_mm': 3,
    'chuva_acum_1h_mm': 6,
    'chuva_acum_3h_mm': 18,
    'chuva_acum_6h_mm': 36,
    'chuva_acum_12h_mm': 72,
    'chuva_acum_24h_mm': 144,
}
for nome_col, n_passos in janelas_chuva.items():
  # Exige janela completa sem NaN para não subestimar chuva acumulada
  feat[nome_col] = chuva.rolling(n_passos, min_periods=n_passos).sum()

# Defasagens (lags) de chuva e maré (10min, 30min, 1h, 2h, 3h, 6h)
lags_passos = [1, 3, 6, 12, 18, 36]
for p in lags_passos:
  minutos = p * 10
  feat[f'chuva_lag_{minutos}min'] = chuva.shift(p)
  feat[f'mare_lag_{minutos}min'] = mare.shift(p)

# Variação da maré em 1h e 3h (tendência de enchente/vazante)
feat['mare_diff_1h'] = mare - mare.shift(6)
feat['mare_diff_3h'] = mare - mare.shift(18)

# 4. Filtrar apenas registros com alvo real da lagoa e todas as features válidas
df_model = feat.copy()
df_model['NA_lagoa_10min_m'] = alvo
df_model['datetime'] = df_model.index
df_valido = df_model.dropna().reset_index(drop=True)

feature_cols = list(feat.columns)
alvo_col = 'NA_lagoa_10min_m'

X = df_valido[feature_cols].values
y = df_valido[alvo_col].values
datas = df_valido['datetime']

# 5. Reutilizar splits salvos se o tamanho coincidir; caso contrário, recria 70/30 + 4 folds
with open(caminho_splits_json, 'r', encoding='utf-8') as f:
  splits = json.load(f)

if splits.get('n_total_validos') == len(df_valido):
  treino_70_idx = splits['treino_70_idx']
  holdout_30_idx = splits['holdout_30_idx']
  folds_4 = splits['folds_4']
  print(
      'Usando partições exatas de splits_70_holdout30_4folds_blocos.json'
      f' ({len(df_valido)} amostras).'
  )
else:
  print(
      'Aviso: total de linhas válidas mudou com acumulados/lags '
      f'({len(df_valido)} vs {splits.get("n_total_validos")}). '
      'Gerando divisão 70/30 + 4 blocos contíguos na base filtrada...'
  )
  n_total = len(df_valido)
  corte_70 = int(n_total * 0.70)
  treino_70_idx = np.arange(0, corte_70).tolist()
  holdout_30_idx = np.arange(corte_70, n_total).tolist()
  kf = KFold(n_splits=4, shuffle=False)
  folds_4 = {}
  for f_num, (tr_p, te_p) in enumerate(kf.split(treino_70_idx), start=1):
    folds_4[f'fold_{f_num}'] = {
        'train_idx': np.array(treino_70_idx)[tr_p].tolist(),
        'test_fold_idx': np.array(treino_70_idx)[te_p].tolist(),
    }

# 6. Treinamento e validação nos 4 k-folds
linhas_folds = []
for fold_num in range(1, 5):
  info_fold = folds_4[f'fold_{fold_num}']
  tr_idx = info_fold['train_idx']
  te_idx = info_fold['test_fold_idx']

  rf_fold = RandomForestRegressor(
      n_estimators=200,
      max_depth=16,
      min_samples_leaf=5,
      n_jobs=-1,
      random_state=42,
  )
  rf_fold.fit(X[tr_idx], y[tr_idx])
  pred_te = rf_fold.predict(X[te_idx])
  m = calcular_metricas_regressao(y[te_idx], pred_te)

  linhas_folds.append({
      'modelo': NOME_MODELO,
      'etapa': f'fold_{fold_num}',
      'n_amostras': len(te_idx),
      'r': round(m['r'], 4) if pd.notna(m['r']) else np.nan,
      'R2': round(m['R2'], 4),
      'SSE': round(m['SSE'], 4),
      'MSE': round(m['MSE'], 6),
  })

df_folds = pd.DataFrame(linhas_folds)
media_4folds = {
    'modelo': NOME_MODELO,
    'etapa': 'media_4folds',
    'n_amostras': int(df_folds['n_amostras'].mean()),
    'r': round(float(df_folds['r'].mean()), 4),
    'R2': round(float(df_folds['R2'].mean()), 4),
    'SSE': round(float(df_folds['SSE'].mean()), 4),
    'MSE': round(float(df_folds['MSE'].mean()), 6),
}

# 7. Treino em 70% e avaliação no Holdout (30%)
rf_70 = RandomForestRegressor(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
rf_70.fit(X[treino_70_idx], y[treino_70_idx])
pred_ho30 = rf_70.predict(X[holdout_30_idx])
m_ho = calcular_metricas_regressao(y[holdout_30_idx], pred_ho30)

linha_holdout = {
    'modelo': NOME_MODELO,
    'etapa': 'holdout_30',
    'n_amostras': len(holdout_30_idx),
    'r': round(m_ho['r'], 4) if pd.notna(m_ho['r']) else np.nan,
    'R2': round(m_ho['R2'], 4),
    'SSE': round(m_ho['SSE'], 4),
    'MSE': round(m_ho['MSE'], 6),
}

df_avaliacao = pd.concat(
    [df_folds, pd.DataFrame([media_4folds, linha_holdout])], ignore_index=True
)

print(
    '=== VALIDAÇÃO DO RANDOM FOREST (10 MIN + ACUMULADOS + LAGS) ==='
)
print(df_avaliacao.to_string(index=False))

# Top 10 variáveis mais importantes
top_feat = pd.DataFrame({
    'variavel': feature_cols,
    'importancia': np.round(rf_70.feature_importances_, 4),
}).sort_values('importancia', ascending=False).head(10)
print('\n=== TOP 10 VARIÁVEIS MAIS IMPORTANTES ===')
print(top_feat.to_string(index=False))

# 8. Salvar resultados e atualizar tabela comparativa geral
caminho_av_atual = os.path.join(
    pasta_comparacao, 'avaliacao_4folds_holdout30_rf_acum_lags.csv'
)
caminho_comp_geral = os.path.join(
    pasta_comparacao, 'comparacao_geral_modelos.csv'
)
caminho_modelo_atual = os.path.join(
    pasta_comparacao, 'modelo_rf_10min_acum_lags_treino70.joblib'
)

df_avaliacao.to_csv(caminho_av_atual, index=False)
joblib.dump(rf_70, caminho_modelo_atual)

if os.path.exists(caminho_comp_geral):
  df_comp_ant = pd.read_csv(caminho_comp_geral)
  df_comp_ant = df_comp_ant[df_comp_ant['modelo'] != NOME_MODELO]
  df_comp_geral = pd.concat([df_comp_ant, df_avaliacao], ignore_index=True)
else:
  df_comp_geral = df_avaliacao.copy()

df_comp_geral.to_csv(caminho_comp_geral, index=False)

resumo_pivot = df_comp_geral[
    df_comp_geral['etapa'].isin(['media_4folds', 'holdout_30'])
].pivot(index='modelo', columns='etapa', values=['r', 'R2', 'SSE', 'MSE'])
print('\n=== COMPARAÇÃO LADO A LADO (MÉDIA 4 FOLDS VS. HOLDOUT 30%) ===')
print(resumo_pivot.to_string())

Mounted at /content/drive
Aviso: total de linhas válidas mudou com acumulados/lags (205736 vs 205879). Gerando divisão 70/30 + 4 blocos contíguos na base filtrada...
=== VALIDAÇÃO DO RANDOM FOREST (10 MIN + ACUMULADOS + LAGS) ===
                               modelo        etapa  n_amostras      r      R2       SSE      MSE
RandomForestRegressor_10min_acum_lags       fold_1       36004 0.6180 -0.0015  756.8052 0.021020
RandomForestRegressor_10min_acum_lags       fold_2       36004 0.6175  0.2542  443.2712 0.012312
RandomForestRegressor_10min_acum_lags       fold_3       36004 0.5988  0.2880  271.1088 0.007530
RandomForestRegressor_10min_acum_lags       fold_4       36003 0.6103 -0.5364  816.4846 0.022678
RandomForestRegressor_10min_acum_lags media_4folds       36003 0.6111  0.0011  571.9174 0.015885
RandomForestRegressor_10min_acum_lags   holdout_30       61721 0.4677  0.1981 2097.2773 0.033980

=== TOP 10 VARIÁVEIS MAIS IMPORTANTES ===
         variavel  importancia
  mare_lag_360min

### **2.3 Modelo de rede neural rasa (Banco de dados com acumulados e defasagem)**

In [2]:
import json
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos de entrada e saída
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_splits = os.path.join(pasta_consolidados, 'SPLITS_70_30_4FOLDS')
pasta_comparacao = os.path.join(
    pasta_consolidados, 'VALIDACAO_COMPARACAO_MODELOS'
)
os.makedirs(pasta_comparacao, exist_ok=True)

caminho_tabela_10min = os.path.join(
    pasta_consolidados, 'tabela_unificada_apenas_10min_ate_20170916_1840.csv'
)
caminho_splits_json = os.path.join(
    pasta_splits, 'splits_70_holdout30_4folds_blocos.json'
)

# Arquitetura rasa com 3 camadas ocultas (ex.: 2 camadas seria (64, 32))
ARQUITETURA_CAMADAS = (64, 32, 16)
NOME_MODELO = f'MLPRegressor_10min_acum_lags_{len(ARQUITETURA_CAMADAS)}camadas'


def calcular_metricas_regressao(y_real, y_pred):
  """Calcula r (Pearson), R², SSE e MSE apenas sobre valores reais válidos."""
  y_real = np.asarray(y_real, dtype=float)
  y_pred = np.asarray(y_pred, dtype=float)

  residuos = y_real - y_pred
  sse = float(np.sum(residuos**2))
  mse = float(mean_squared_error(y_real, y_pred))
  r2 = float(r2_score(y_real, y_pred))

  if np.std(y_real) == 0 or np.std(y_pred) == 0:
    r = np.nan
  else:
    r = float(np.corrcoef(y_real, y_pred)[0, 1])

  return {'r': r, 'R2': r2, 'SSE': sse, 'MSE': mse}


def criar_mlp_raso():
  """Retorna rede neural rasa configurada com 2 a 3 camadas ocultas."""
  return MLPRegressor(
      hidden_layer_sizes=ARQUITETURA_CAMADAS,
      activation='relu',
      solver='adam',
      alpha=1e-3,
      batch_size=256,
      learning_rate='adaptive',
      learning_rate_init=1e-3,
      max_iter=150,
      early_stopping=True,
      validation_fraction=0.1,
      n_iter_no_change=12,
      random_state=42,
      verbose=False,
  )


# 2. Carregar tabela unificada de 10 min e garantir grade contínua para acumulados/lags
df = pd.read_csv(caminho_tabela_10min)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = (
    df.dropna(subset=['datetime'])
    .sort_values('datetime')
    .drop_duplicates(subset=['datetime'], keep='last')
    .set_index('datetime')
)

grade_10min = pd.date_range(
    start=df.index.min(), end=df.index.max(), freq='10min'
)
df_grade = df.reindex(grade_10min)

chuva = pd.to_numeric(df_grade['chuva_10min_mm'], errors='coerce')
mare = pd.to_numeric(df_grade['mare_10min_m'], errors='coerce')
alvo = pd.to_numeric(df_grade['NA_lagoa_10min_m'], errors='coerce')

# 3. Mesmas variáveis: instantâneas de 10 min + acumulados de chuva + defasagens (lags)
feat = pd.DataFrame(index=grade_10min)
feat['chuva_10min_mm'] = chuva
feat['mare_10min_m'] = mare

janelas_chuva = {
    'chuva_acum_30min_mm': 3,
    'chuva_acum_1h_mm': 6,
    'chuva_acum_3h_mm': 18,
    'chuva_acum_6h_mm': 36,
    'chuva_acum_12h_mm': 72,
    'chuva_acum_24h_mm': 144,
}
for nome_col, n_passos in janelas_chuva.items():
  feat[nome_col] = chuva.rolling(n_passos, min_periods=n_passos).sum()

lags_passos = [1, 3, 6, 12, 18, 36]
for p in lags_passos:
  minutos = p * 10
  feat[f'chuva_lag_{minutos}min'] = chuva.shift(p)
  feat[f'mare_lag_{minutos}min'] = mare.shift(p)

feat['mare_diff_1h'] = mare - mare.shift(6)
feat['mare_diff_3h'] = mare - mare.shift(18)

# 4. Filtrar apenas registros com alvo real da lagoa e todas as features válidas
df_model = feat.copy()
df_model['NA_lagoa_10min_m'] = alvo
df_model['datetime'] = df_model.index
df_valido = df_model.dropna().reset_index(drop=True)

feature_cols = list(feat.columns)
alvo_col = 'NA_lagoa_10min_m'

X = df_valido[feature_cols].values
y = df_valido[alvo_col].values

# 5. Reutilizar splits salvos se o tamanho coincidir; caso contrário, recria 70/30 + 4 folds
with open(caminho_splits_json, 'r', encoding='utf-8') as f:
  splits = json.load(f)

if splits.get('n_total_validos') == len(df_valido):
  treino_70_idx = splits['treino_70_idx']
  holdout_30_idx = splits['holdout_30_idx']
  folds_4 = splits['folds_4']
  print(
      'Usando partições exatas de splits_70_holdout30_4folds_blocos.json'
      f' ({len(df_valido)} amostras).'
  )
else:
  print(
      'Aviso: total de linhas válidas mudou com acumulados/lags '
      f'({len(df_valido)} vs {splits.get("n_total_validos")}). '
      'Gerando divisão 70/30 + 4 blocos contíguos na base filtrada...'
  )
  n_total = len(df_valido)
  corte_70 = int(n_total * 0.70)
  treino_70_idx = np.arange(0, corte_70).tolist()
  holdout_30_idx = np.arange(corte_70, n_total).tolist()
  kf = KFold(n_splits=4, shuffle=False)
  folds_4 = {}
  for f_num, (tr_p, te_p) in enumerate(kf.split(treino_70_idx), start=1):
    folds_4[f'fold_{f_num}'] = {
        'train_idx': np.array(treino_70_idx)[tr_p].tolist(),
        'test_fold_idx': np.array(treino_70_idx)[te_p].tolist(),
    }

# 6. Treinamento e validação nos 4 k-folds (com scaler ajustado apenas no treino de cada fold)
linhas_folds = []
for fold_num in range(1, 5):
  info_fold = folds_4[f'fold_{fold_num}']
  tr_idx = info_fold['train_idx']
  te_idx = info_fold['test_fold_idx']

  scaler_fold = StandardScaler()
  X_tr_scaled = scaler_fold.fit_transform(X[tr_idx])
  X_te_scaled = scaler_fold.transform(X[te_idx])

  mlp_fold = criar_mlp_raso()
  mlp_fold.fit(X_tr_scaled, y[tr_idx])
  pred_te = mlp_fold.predict(X_te_scaled)

  m = calcular_metricas_regressao(y[te_idx], pred_te)
  linhas_folds.append({
      'modelo': NOME_MODELO,
      'etapa': f'fold_{fold_num}',
      'n_amostras': len(te_idx),
      'r': round(m['r'], 4) if pd.notna(m['r']) else np.nan,
      'R2': round(m['R2'], 4),
      'SSE': round(m['SSE'], 4),
      'MSE': round(m['MSE'], 6),
  })

df_folds = pd.DataFrame(linhas_folds)
media_4folds = {
    'modelo': NOME_MODELO,
    'etapa': 'media_4folds',
    'n_amostras': int(df_folds['n_amostras'].mean()),
    'r': round(float(df_folds['r'].mean()), 4),
    'R2': round(float(df_folds['R2'].mean()), 4),
    'SSE': round(float(df_folds['SSE'].mean()), 4),
    'MSE': round(float(df_folds['MSE'].mean()), 6),
}

# 7. Treino em 70% e avaliação no Holdout (30%)
scaler_70 = StandardScaler()
X_tr70_scaled = scaler_70.fit_transform(X[treino_70_idx])
X_ho30_scaled = scaler_70.transform(X[holdout_30_idx])

mlp_70 = criar_mlp_raso()
mlp_70.fit(X_tr70_scaled, y[treino_70_idx])
pred_ho30 = mlp_70.predict(X_ho30_scaled)

m_ho = calcular_metricas_regressao(y[holdout_30_idx], pred_ho30)
linha_holdout = {
    'modelo': NOME_MODELO,
    'etapa': 'holdout_30',
    'n_amostras': len(holdout_30_idx),
    'r': round(m_ho['r'], 4) if pd.notna(m_ho['r']) else np.nan,
    'R2': round(m_ho['R2'], 4),
    'SSE': round(m_ho['SSE'], 4),
    'MSE': round(m_ho['MSE'], 6),
}

df_avaliacao_rna = pd.concat(
    [df_folds, pd.DataFrame([media_4folds, linha_holdout])], ignore_index=True
)

print(
    '=== VALIDAÇÃO DA REDE NEURAL RASA'
    f' {ARQUITETURA_CAMADAS} (4 FOLDS + HOLDOUT 30%) ==='
)
print(df_avaliacao_rna.to_string(index=False))

# 8. Salvar resultados específicos da rede neural (sem sobrescrever arquivos do RF)
caminho_av_rna = os.path.join(
    pasta_comparacao,
    f'avaliacao_4folds_holdout30_rna_{len(ARQUITETURA_CAMADAS)}camadas.csv',
)
caminho_modelo_rna = os.path.join(
    pasta_comparacao,
    f'modelo_rna_{len(ARQUITETURA_CAMADAS)}camadas_10min_acum_lags_treino70.joblib',
)

df_avaliacao_rna.to_csv(caminho_av_rna, index=False)
joblib.dump(
    {'modelo': mlp_70, 'scaler': scaler_70, 'features': feature_cols},
    caminho_modelo_rna,
)

print('\nArquivos da rede neural salvos em:', pasta_comparacao)
print(f'  • {caminho_av_rna}')
print(f'  • {caminho_modelo_rna}')

Aviso: total de linhas válidas mudou com acumulados/lags (205736 vs 205879). Gerando divisão 70/30 + 4 blocos contíguos na base filtrada...


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(


=== VALIDAÇÃO DA REDE NEURAL RASA (64, 32, 16) (4 FOLDS + HOLDOUT 30%) ===
                               modelo        etapa  n_amostras      r      R2       SSE      MSE
MLPRegressor_10min_acum_lags_3camadas       fold_1       36004 0.6440  0.0816  694.0266 0.019276
MLPRegressor_10min_acum_lags_3camadas       fold_2       36004 0.6244  0.2575  441.3509 0.012258
MLPRegressor_10min_acum_lags_3camadas       fold_3       36004 0.6065  0.2794  274.3917 0.007621
MLPRegressor_10min_acum_lags_3camadas       fold_4       36003 0.6273 -0.5590  828.5085 0.023012
MLPRegressor_10min_acum_lags_3camadas media_4folds       36003 0.6256  0.0149  559.5694 0.015542
MLPRegressor_10min_acum_lags_3camadas   holdout_30       61721 0.4862  0.2111 2063.3075 0.033430

Arquivos da rede neural salvos em: /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/VALIDACAO_COMPARACAO_MODELOS
  • /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/VALIDACAO_COMPARACAO_MODELOS/avaliacao_4folds_holdout30_rna_3camadas.csv
  • /con

rede neural com max_iter = 350

In [3]:
import json
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos de entrada e saída
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_splits = os.path.join(pasta_consolidados, 'SPLITS_70_30_4FOLDS')
pasta_comparacao = os.path.join(
    pasta_consolidados, 'VALIDACAO_COMPARACAO_MODELOS'
)
os.makedirs(pasta_comparacao, exist_ok=True)

caminho_tabela_10min = os.path.join(
    pasta_consolidados, 'tabela_unificada_apenas_10min_ate_20170916_1840.csv'
)
caminho_splits_json = os.path.join(
    pasta_splits, 'splits_70_holdout30_4folds_blocos.json'
)

ARQUITETURA_CAMADAS = (64, 32, 16)
MAX_ITER = 350
NOME_MODELO = (
    f'MLPRegressor_10min_acum_lags_{len(ARQUITETURA_CAMADAS)}camadas_iter{MAX_ITER}'
)


def calcular_metricas_regressao(y_real, y_pred):
  """Calcula r (Pearson), R², SSE e MSE apenas sobre valores reais válidos."""
  y_real = np.asarray(y_real, dtype=float)
  y_pred = np.asarray(y_pred, dtype=float)

  residuos = y_real - y_pred
  sse = float(np.sum(residuos**2))
  mse = float(mean_squared_error(y_real, y_pred))
  r2 = float(r2_score(y_real, y_pred))

  if np.std(y_real) == 0 or np.std(y_pred) == 0:
    r = np.nan
  else:
    r = float(np.corrcoef(y_real, y_pred)[0, 1])

  return {'r': r, 'R2': r2, 'SSE': sse, 'MSE': mse}


def criar_mlp_raso():
  """Retorna rede neural rasa com teto maior de iterações e parada antecipada."""
  return MLPRegressor(
      hidden_layer_sizes=ARQUITETURA_CAMADAS,
      activation='relu',
      solver='adam',
      alpha=1e-3,
      batch_size=256,
      learning_rate='adaptive',
      learning_rate_init=1e-3,
      max_iter=MAX_ITER,
      early_stopping=True,
      validation_fraction=0.1,
      n_iter_no_change=12,
      random_state=42,
      verbose=False,
  )


# 2. Carregar tabela unificada de 10 min e garantir grade contínua para acumulados/lags
df = pd.read_csv(caminho_tabela_10min)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = (
    df.dropna(subset=['datetime'])
    .sort_values('datetime')
    .drop_duplicates(subset=['datetime'], keep='last')
    .set_index('datetime')
)

grade_10min = pd.date_range(
    start=df.index.min(), end=df.index.max(), freq='10min'
)
df_grade = df.reindex(grade_10min)

chuva = pd.to_numeric(df_grade['chuva_10min_mm'], errors='coerce')
mare = pd.to_numeric(df_grade['mare_10min_m'], errors='coerce')
alvo = pd.to_numeric(df_grade['NA_lagoa_10min_m'], errors='coerce')

# 3. Mesmas variáveis: instantâneas de 10 min + acumulados de chuva + defasagens (lags)
feat = pd.DataFrame(index=grade_10min)
feat['chuva_10min_mm'] = chuva
feat['mare_10min_m'] = mare

janelas_chuva = {
    'chuva_acum_30min_mm': 3,
    'chuva_acum_1h_mm': 6,
    'chuva_acum_3h_mm': 18,
    'chuva_acum_6h_mm': 36,
    'chuva_acum_12h_mm': 72,
    'chuva_acum_24h_mm': 144,
}
for nome_col, n_passos in janelas_chuva.items():
  feat[nome_col] = chuva.rolling(n_passos, min_periods=n_passos).sum()

lags_passos = [1, 3, 6, 12, 18, 36]
for p in lags_passos:
  minutos = p * 10
  feat[f'chuva_lag_{minutos}min'] = chuva.shift(p)
  feat[f'mare_lag_{minutos}min'] = mare.shift(p)

feat['mare_diff_1h'] = mare - mare.shift(6)
feat['mare_diff_3h'] = mare - mare.shift(18)

# 4. Filtrar apenas registros com alvo real da lagoa e todas as features válidas
df_model = feat.copy()
df_model['NA_lagoa_10min_m'] = alvo
df_model['datetime'] = df_model.index
df_valido = df_model.dropna().reset_index(drop=True)

feature_cols = list(feat.columns)
alvo_col = 'NA_lagoa_10min_m'

X = df_valido[feature_cols].values
y = df_valido[alvo_col].values

# 5. Reutilizar splits salvos se o tamanho coincidir; caso contrário, recria 70/30 + 4 folds
with open(caminho_splits_json, 'r', encoding='utf-8') as f:
  splits = json.load(f)

if splits.get('n_total_validos') == len(df_valido):
  treino_70_idx = splits['treino_70_idx']
  holdout_30_idx = splits['holdout_30_idx']
  folds_4 = splits['folds_4']
  print(
      'Usando partições exatas de splits_70_holdout30_4folds_blocos.json'
      f' ({len(df_valido)} amostras).'
  )
else:
  print(
      'Aviso: total de linhas válidas mudou com acumulados/lags '
      f'({len(df_valido)} vs {splits.get("n_total_validos")}). '
      'Gerando divisão 70/30 + 4 blocos contíguos na base filtrada...'
  )
  n_total = len(df_valido)
  corte_70 = int(n_total * 0.70)
  treino_70_idx = np.arange(0, corte_70).tolist()
  holdout_30_idx = np.arange(corte_70, n_total).tolist()
  kf = KFold(n_splits=4, shuffle=False)
  folds_4 = {}
  for f_num, (tr_p, te_p) in enumerate(kf.split(treino_70_idx), start=1):
    folds_4[f'fold_{f_num}'] = {
        'train_idx': np.array(treino_70_idx)[tr_p].tolist(),
        'test_fold_idx': np.array(treino_70_idx)[te_p].tolist(),
    }

# 6. Treinamento e validação nos 4 k-folds
linhas_folds = []
for fold_num in range(1, 5):
  info_fold = folds_4[f'fold_{fold_num}']
  tr_idx = info_fold['train_idx']
  te_idx = info_fold['test_fold_idx']

  scaler_fold = StandardScaler()
  X_tr_scaled = scaler_fold.fit_transform(X[tr_idx])
  X_te_scaled = scaler_fold.transform(X[te_idx])

  mlp_fold = criar_mlp_raso()
  mlp_fold.fit(X_tr_scaled, y[tr_idx])
  pred_te = mlp_fold.predict(X_te_scaled)

  m = calcular_metricas_regressao(y[te_idx], pred_te)
  linhas_folds.append({
      'modelo': NOME_MODELO,
      'etapa': f'fold_{fold_num}',
      'n_iter_executadas': mlp_fold.n_iter_,
      'n_amostras': len(te_idx),
      'r': round(m['r'], 4) if pd.notna(m['r']) else np.nan,
      'R2': round(m['R2'], 4),
      'SSE': round(m['SSE'], 4),
      'MSE': round(m['MSE'], 6),
  })

df_folds = pd.DataFrame(linhas_folds)
media_4folds = {
    'modelo': NOME_MODELO,
    'etapa': 'media_4folds',
    'n_iter_executadas': round(float(df_folds['n_iter_executadas'].mean()), 1),
    'n_amostras': int(df_folds['n_amostras'].mean()),
    'r': round(float(df_folds['r'].mean()), 4),
    'R2': round(float(df_folds['R2'].mean()), 4),
    'SSE': round(float(df_folds['SSE'].mean()), 4),
    'MSE': round(float(df_folds['MSE'].mean()), 6),
}

# 7. Treino em 70% e avaliação no Holdout (30%)
scaler_70 = StandardScaler()
X_tr70_scaled = scaler_70.fit_transform(X[treino_70_idx])
X_ho30_scaled = scaler_70.transform(X[holdout_30_idx])

mlp_70 = criar_mlp_raso()
mlp_70.fit(X_tr70_scaled, y[treino_70_idx])
pred_ho30 = mlp_70.predict(X_ho30_scaled)

m_ho = calcular_metricas_regressao(y[holdout_30_idx], pred_ho30)
linha_holdout = {
    'modelo': NOME_MODELO,
    'etapa': 'holdout_30',
    'n_iter_executadas': mlp_70.n_iter_,
    'n_amostras': len(holdout_30_idx),
    'r': round(m_ho['r'], 4) if pd.notna(m_ho['r']) else np.nan,
    'R2': round(m_ho['R2'], 4),
    'SSE': round(m_ho['SSE'], 4),
    'MSE': round(m_ho['MSE'], 6),
}

df_avaliacao_rna = pd.concat(
    [df_folds, pd.DataFrame([media_4folds, linha_holdout])], ignore_index=True
)

print(
    '=== VALIDAÇÃO DA REDE NEURAL RASA'
    f' {ARQUITETURA_CAMADAS} (max_iter={MAX_ITER}) ==='
)
print(df_avaliacao_rna.to_string(index=False))

# 8. Salvar resultados específicos desta rodada
caminho_av_rna = os.path.join(
    pasta_comparacao,
    f'avaliacao_4folds_holdout30_rna_{len(ARQUITETURA_CAMADAS)}camadas_iter{MAX_ITER}.csv',
)
caminho_modelo_rna = os.path.join(
    pasta_comparacao,
    f'modelo_rna_{len(ARQUITETURA_CAMADAS)}camadas_iter{MAX_ITER}_treino70.joblib',
)

df_avaliacao_rna.to_csv(caminho_av_rna, index=False)
joblib.dump(
    {'modelo': mlp_70, 'scaler': scaler_70, 'features': feature_cols},
    caminho_modelo_rna,
)

print('\nArquivos salvos em:', pasta_comparacao)
print(f'  • {caminho_av_rna}')
print(f'  • {caminho_modelo_rna}')

Aviso: total de linhas válidas mudou com acumulados/lags (205736 vs 205879). Gerando divisão 70/30 + 4 blocos contíguos na base filtrada...
=== VALIDAÇÃO DA REDE NEURAL RASA (64, 32, 16) (max_iter=350) ===
                                       modelo        etapa  n_iter_executadas  n_amostras      r      R2       SSE      MSE
MLPRegressor_10min_acum_lags_3camadas_iter350       fold_1              121.0       36004 0.6440  0.0816  694.0266 0.019276
MLPRegressor_10min_acum_lags_3camadas_iter350       fold_2              140.0       36004 0.6244  0.2575  441.3509 0.012258
MLPRegressor_10min_acum_lags_3camadas_iter350       fold_3              154.0       36004 0.6065  0.2794  274.3917 0.007621
MLPRegressor_10min_acum_lags_3camadas_iter350       fold_4              105.0       36003 0.6273 -0.5590  828.5085 0.023012
MLPRegressor_10min_acum_lags_3camadas_iter350 media_4folds              130.0       36003 0.6256  0.0149  559.5694 0.015542
MLPRegressor_10min_acum_lags_3camadas_iter350   ho

Análise de métricas da rede neural

In [6]:
import joblib

pacote = joblib.load(caminho_modelo_rna)
mlp_70 = pacote['modelo']
scaler_70 = pacote['scaler']
feature_cols = pacote['features']

X_ho30 = df_valido.loc[holdout_30_idx, feature_cols].values
y_ho30 = df_valido.loc[holdout_30_idx, 'NA_lagoa_10min_m'].values

X_ho30_scaled = scaler_70.transform(X_ho30)
pred_ho30 = mlp_70.predict(X_ho30_scaled)

resumo_diferencas_e_variacao('holdout_30', y_ho30, pred_ho30)

--- holdout_30 ---
  Diferença média (previsto - observado / bias): -0.028939 m
  Erro absoluto médio (MAE):                     0.111075 m
  Raiz do erro quadrático médio (RMSE):          0.182838 m
  Desvio padrão do erro (dispersão):           0.180534 m
  Variação média |Δy| real (10 min):             0.003673 m
  Variação média |Δŷ| prevista (10 min):           0.006361 m


### **2.4 Treinamento final com rede neural**

In [7]:
import json
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

try:
  from google.colab import drive

  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
except Exception:
  pass

# 1. Caminhos de entrada e saída
pasta_base = '/content/drive/MyDrive/PUC/EQM2118'
pasta_consolidados = os.path.join(pasta_base, 'CONSOLIDADOS')
pasta_modelo_final = os.path.join(pasta_consolidados, 'MODELO_FINAL_RNA')
os.makedirs(pasta_modelo_final, exist_ok=True)

caminho_tabela_10min = os.path.join(
    pasta_consolidados, 'tabela_unificada_apenas_10min_ate_20170916_1840.csv'
)

ARQUITETURA_CAMADAS = (64, 32, 16)
MAX_ITER = 350
NOME_MODELO_FINAL = (
    f'MLPRegressor_10min_acum_lags_{len(ARQUITETURA_CAMADAS)}camadas_iter{MAX_ITER}_FINAL'
)


def calcular_metricas_ajuste(y_real, y_pred):
  """Calcula métricas de desempenho sobre observações reais válidas."""
  y_real = np.asarray(y_real, dtype=float)
  y_pred = np.asarray(y_pred, dtype=float)

  erro = y_pred - y_real
  sse = float(np.sum((y_real - y_pred) ** 2))
  mse = float(mean_squared_error(y_real, y_pred))
  rmse = float(np.sqrt(mse))
  mae = float(mean_absolute_error(y_real, y_pred))
  bias = float(np.mean(erro))
  r2 = float(r2_score(y_real, y_pred))

  if np.std(y_real) == 0 or np.std(y_pred) == 0:
    r = np.nan
  else:
    r = float(np.corrcoef(y_real, y_pred)[0, 1])

  return {
      'r': r,
      'R2': r2,
      'SSE': sse,
      'MSE': mse,
      'RMSE': rmse,
      'MAE': mae,
      'bias_m': bias,
  }


def criar_mlp_final():
  """Cria a rede neural rasa com os parâmetros definidos na validação."""
  return MLPRegressor(
      hidden_layer_sizes=ARQUITETURA_CAMADAS,
      activation='relu',
      solver='adam',
      alpha=1e-3,
      batch_size=256,
      learning_rate='adaptive',
      learning_rate_init=1e-3,
      max_iter=MAX_ITER,
      early_stopping=True,
      validation_fraction=0.1,
      n_iter_no_change=12,
      random_state=42,
      verbose=False,
  )


# 2. Carregar tabela unificada de 10 min e alinhar na grade contínua sem interpolar
df = pd.read_csv(caminho_tabela_10min)
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = (
    df.dropna(subset=['datetime'])
    .sort_values('datetime')
    .drop_duplicates(subset=['datetime'], keep='last')
    .set_index('datetime')
)

grade_10min = pd.date_range(
    start=df.index.min(), end=df.index.max(), freq='10min'
)
df_grade = df.reindex(grade_10min)

chuva = pd.to_numeric(df_grade['chuva_10min_mm'], errors='coerce')
mare = pd.to_numeric(df_grade['mare_10min_m'], errors='coerce')
alvo = pd.to_numeric(df_grade['NA_lagoa_10min_m'], errors='coerce')

# 3. Construir variáveis preditoras (instantâneas + acumulados de chuva + defasagens)
feat = pd.DataFrame(index=grade_10min)
feat['chuva_10min_mm'] = chuva
feat['mare_10min_m'] = mare

janelas_chuva = {
    'chuva_acum_30min_mm': 3,
    'chuva_acum_1h_mm': 6,
    'chuva_acum_3h_mm': 18,
    'chuva_acum_6h_mm': 36,
    'chuva_acum_12h_mm': 72,
    'chuva_acum_24h_mm': 144,
}
for nome_col, n_passos in janelas_chuva.items():
  feat[nome_col] = chuva.rolling(n_passos, min_periods=n_passos).sum()

lags_passos = [1, 3, 6, 12, 18, 36]
for p in lags_passos:
  minutos = p * 10
  feat[f'chuva_lag_{minutos}min'] = chuva.shift(p)
  feat[f'mare_lag_{minutos}min'] = mare.shift(p)

feat['mare_diff_1h'] = mare - mare.shift(6)
feat['mare_diff_3h'] = mare - mare.shift(18)

# 4. Filtrar registros válidos sem interpolação
df_model = feat.copy()
df_model['NA_lagoa_10min_m'] = alvo
df_valido = df_model.dropna().copy()

feature_cols = list(feat.columns)
alvo_col = 'NA_lagoa_10min_m'

X_full = df_valido[feature_cols].values
y_full = df_valido[alvo_col].values

# 5. Padronizar entradas e treinar modelo final em toda a base válida até 16/09/2017 18:40
scaler_final = StandardScaler()
X_full_scaled = scaler_final.fit_transform(X_full)

mlp_final = criar_mlp_final()
mlp_final.fit(X_full_scaled, y_full)

y_pred_treino = mlp_final.predict(X_full_scaled)
metricas = calcular_metricas_ajuste(y_full, y_pred_treino)

resumo = {
    'modelo': NOME_MODELO_FINAL,
    'arquitetura': str(ARQUITETURA_CAMADAS),
    'max_iter': MAX_ITER,
    'n_iter_executadas': int(mlp_final.n_iter_),
    'n_amostras_treino_completo': int(len(df_valido)),
    'data_inicio_treino': str(df_valido.index.min()),
    'data_fim_treino': str(df_valido.index.max()),
    'r': round(metricas['r'], 4) if pd.notna(metricas['r']) else np.nan,
    'R2': round(metricas['R2'], 4),
    'SSE': round(metricas['SSE'], 4),
    'MSE': round(metricas['MSE'], 6),
    'RMSE_m': round(metricas['RMSE'], 6),
    'MAE_m': round(metricas['MAE'], 6),
    'bias_m': round(metricas['bias_m'], 6),
}

print('=== TREINO FINAL CONCLUÍDO ===')
for k, v in resumo.items():
  print(f'  • {k}: {v}')

# 6. Salvar pacote completo do modelo final e resumo
caminho_pacote_final = os.path.join(
    pasta_modelo_final, f'{NOME_MODELO_FINAL}.joblib'
)
caminho_resumo_json = os.path.join(
    pasta_modelo_final, f'{NOME_MODELO_FINAL}_resumo.json'
)

joblib.dump(
    {
        'modelo': mlp_final,
        'scaler': scaler_final,
        'features': feature_cols,
        'alvo': alvo_col,
        'resumo': resumo,
    },
    caminho_pacote_final,
)

with open(caminho_resumo_json, 'w', encoding='utf-8') as f:
  json.dump(resumo, f, indent=2, ensure_ascii=False)

print('\nArquivos salvos em:', pasta_modelo_final)
print(f'  • {caminho_pacote_final}')
print(f'  • {caminho_resumo_json}')

=== TREINO FINAL CONCLUÍDO ===
  • modelo: MLPRegressor_10min_acum_lags_3camadas_iter350_FINAL
  • arquitetura: (64, 32, 16)
  • max_iter: 350
  • n_iter_executadas: 129
  • n_amostras_treino_completo: 205736
  • data_inicio_treino: 2013-03-01 16:10:00
  • data_fim_treino: 2017-09-16 18:40:00
  • r: 0.6594
  • R2: 0.4349
  • SSE: 3091.9169
  • MSE: 0.015029
  • RMSE_m: 0.122591
  • MAE_m: 0.084455
  • bias_m: -0.000429

Arquivos salvos em: /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/MODELO_FINAL_RNA
  • /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/MODELO_FINAL_RNA/MLPRegressor_10min_acum_lags_3camadas_iter350_FINAL.joblib
  • /content/drive/MyDrive/PUC/EQM2118/CONSOLIDADOS/MODELO_FINAL_RNA/MLPRegressor_10min_acum_lags_3camadas_iter350_FINAL_resumo.json
